## 📊 Notebook-Übung: Tabellen verknüpfen mit SQL JOINs

:::{.callout-note}
## Wie man dieses Notebook lokal ausführt

1. Erstelle einen Ordner
2. Erstelle eine virtuelle Python-Umgebung  
   `python -m venv .venv`
3. Aktiviere die virtuelle Umgebung  
   `.venv\Scripts\activate`
4. Installiere Jupyter und pandas  
   `pip install jupyter pandas`
5. Lade diese Datei in deinen Ordner herunter
6. Starte den lokalen Jupyter-Server  
   `jupyter-lab`
:::

### Kontext

In der realen Welt sind Daten fast nie in einer einzigen Tabelle gespeichert.
Kundendaten liegen getrennt von Bestelldaten, Klassenlisten getrennt von Noten.
Um sinnvolle Analysen zu machen, müssen wir Tabellen **verknüpfen** — das
nennt man einen **JOIN**.

### Das Wichtigste auf einen Blick

| Was ich will | SQL |
|---|---|
| Nur übereinstimmende Zeilen | `SELECT ... FROM A INNER JOIN B ON A.key = B.key` |
| Alle Zeilen der linken Tabelle | `SELECT ... FROM A LEFT JOIN B ON A.key = B.key` |
| Fehlende Einträge finden | `... LEFT JOIN ... WHERE B.spalte IS NULL` |

**Dauer:** ca. 40–50 Minuten

**Voraussetzungen:** SQL-Grundlagen (`SELECT`, `FROM`, `WHERE`, Aggregatfunktionen), grundlegende Python-Kenntnisse

---

### Setup

Führe die folgende Zelle aus, um die Datenbank vorzubereiten. **Nicht verändern.**

In [ ]:
# --- Setup (nicht verändern) ---
import sqlite3
import pandas as pd

# Hilfsfunktion: Führt eine SQL-Abfrage aus und gibt ein DataFrame zurück.
def sql(query):
    cursor.execute(query)
    cols = [d[0] for d in cursor.description]
    return pd.DataFrame(cursor.fetchall(), columns=cols)

# Erstelle eine SQLite-Datenbank im Arbeitsspeicher (RAM)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript("""
-- Tabellen für Aufgaben 1-2: Die Prüfungsliste
CREATE TABLE klassenliste (
    schueler_id INTEGER PRIMARY KEY,
    name        TEXT,
    klasse      TEXT
);
CREATE TABLE noten (
    schueler_id INTEGER,
    note        REAL,
    FOREIGN KEY (schueler_id) REFERENCES klassenliste(schueler_id)
);
INSERT INTO klassenliste VALUES
    (1, 'Lea',   '2fP'),
    (2, 'Jan',   '2fP'),
    (3, 'Mira',  '2fP'),
    (4, 'Noah',  '2fP'),
    (5, 'Elena', '2fP'),
    (6, 'Luca',  '2fP');
INSERT INTO noten VALUES
    (1, 5.5),
    (2, 4.0),
    (5, 5.0);

-- Tabellen für Aufgabe 3: Der Online-Shop (gleiche Daten wie im SQL-Aggregatfunktionen-Notebook!)
CREATE TABLE products (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT,
    category     TEXT,
    price        REAL
);
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    product_id  INTEGER,
    quantity    INTEGER,
    customer    TEXT,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
INSERT INTO products VALUES
    (1, 'Laptop',  'Electronics', 1200.00),
    (2, 'Mouse',   'Electronics',   25.00),
    (3, 'Desk',    'Furniture',    450.00),
    (4, 'Chair',   'Furniture',    320.00),
    (5, 'Headset', 'Electronics',   80.00);
INSERT INTO orders VALUES
    (1, 1, 2, 'Anna'),
    (2, 2, 5, 'Ben'),
    (3, 3, 1, 'Clara'),
    (4, 1, 1, 'Daniel'),
    (5, 4, 3, 'Anna'),
    (6, 5, 2, 'Ben'),
    (7, 2, 8, 'Clara'),
    (8, 3, 2, 'Evi'),
    (9, 5, 1, 'Anna');

-- Tabellen für Aufgabe 4: Sportverein
CREATE TABLE mitglieder (
    mitglied_id INTEGER PRIMARY KEY,
    name        TEXT,
    sportart    TEXT
);
CREATE TABLE turnier_ergebnisse (
    mitglied_id INTEGER,
    turnier     TEXT,
    rang        INTEGER,
    FOREIGN KEY (mitglied_id) REFERENCES mitglieder(mitglied_id)
);
INSERT INTO mitglieder VALUES
    (1, 'Anna',   'Fussball'),
    (2, 'Ben',    'Tennis'),
    (3, 'Chiara', 'Fussball'),
    (4, 'David',  'Schwimmen'),
    (5, 'Eva',    'Tennis'),
    (6, 'Finn',   'Fussball');
INSERT INTO turnier_ergebnisse VALUES
    (1, 'Kantonalmeisterschaft', 2),
    (2, 'Vereinsturnier', 5),
    (3, 'Kantonalmeisterschaft', 8),
    (5, 'Vereinsturnier', 3);

-- Tabellen für Aufgabe 5: Bibliothek
CREATE TABLE buecher (
    buch_id INTEGER PRIMARY KEY,
    titel   TEXT,
    autor   TEXT,
    genre   TEXT
);
CREATE TABLE ausleihen (
    ausleihe_id INTEGER PRIMARY KEY,
    buch_id     INTEGER,
    schueler    TEXT,
    datum       TEXT,
    FOREIGN KEY (buch_id) REFERENCES buecher(buch_id)
);
INSERT INTO buecher VALUES
    (1, 'Harry Potter', 'J.K. Rowling', 'Fantasy'),
    (2, 'Gregs Tagebuch', 'Jeff Kinney', 'Humor'),
    (3, 'Die drei ???', 'Verschiedene', 'Krimi'),
    (4, 'Eragon', 'C. Paolini', 'Fantasy'),
    (5, 'Tintenherz', 'C. Funke', 'Fantasy'),
    (6, 'Rico, Oskar und die Tieferschatten', 'A. Steinhöfel', 'Krimi');
INSERT INTO ausleihen VALUES
    (1, 1, 'Lea',   '2026-03-10'),
    (2, 2, 'Jan',   '2026-03-12'),
    (3, 1, 'Mira',  '2026-03-15'),
    (4, 3, 'Noah',  '2026-03-18'),
    (5, 2, 'Elena', '2026-03-20');
""")
conn.commit()
print('Datenbank bereit. Viel Erfolg!')

---

### Aufgabe 1: INNER JOIN — Wer hat die Prüfung geschrieben?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `klassenliste` | `schueler_id`, `name`, `klasse` |
| `noten` | `schueler_id`, `note` |

Frau Müller will nur die Schüler sehen, die die Prüfung **tatsächlich
geschrieben haben**, zusammen mit ihren Namen und Noten.

Schreibe ein `SELECT` mit `INNER JOIN`, um `klassenliste` und `noten`
über die Spalte `schueler_id` zu verknüpfen. Zeige `name` und `note`.

**Erwartetes Ergebnis:** 3 Zeilen (Lea, Jan, Elena).

In [ ]:
# AUFGABE 1: INNER JOIN von klassenliste und noten.
# Nutze sql(""" ... """) um deine Abfrage auszuführen.

result1 = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 1:\n', result1)

<details>
<summary>Lösung anzeigen</summary>

```sql
SELECT klassenliste.name, noten.note
FROM klassenliste
INNER JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id;
```

Ergebnis: 3 Zeilen — nur Lea (5.5), Jan (4.0) und Elena (5.0), weil nur diese in **beiden** Tabellen vorkommen.

</details>

### Aufgabe 2: LEFT JOIN — Die vollständige Übersicht

Frau Müller braucht **alle 6 Schüler** in ihrer Übersicht — auch die,
die nicht an der Prüfung teilgenommen haben. Bei fehlenden Noten
soll `NULL` stehen.

**2a)** Schreibe ein `SELECT` mit `LEFT JOIN`, um alle Schüler mit ihren
Noten zu sehen.

**2b)** Erweitere die Abfrage mit einer `WHERE`-Klausel, um nur die
Schüler zu finden, die **keine Note** haben (`IS NULL`).

**2c)** Beantworte: Warum stehen dort `NULL`-Werte?

In [ ]:
# AUFGABE 2a: LEFT JOIN von klassenliste und noten.

result2a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 2a:\n', result2a)

<details>
<summary>Lösung anzeigen (2a)</summary>

```sql
SELECT klassenliste.name, noten.note
FROM klassenliste
LEFT JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id;
```

Ergebnis: 6 Zeilen. Mira, Noah und Luca haben `None` (= NULL) in der Notenspalte, weil sie nicht in der `noten`-Tabelle vorkommen.

</details>

In [ ]:
# AUFGABE 2b: Nur Schüler OHNE Note finden (LEFT JOIN + WHERE IS NULL).

result2b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 2b:\n', result2b)

<details>
<summary>Lösung anzeigen (2b)</summary>

```sql
SELECT klassenliste.name
FROM klassenliste
LEFT JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id
WHERE noten.note IS NULL;
```

Ausgabe: Mira, Noah, Luca — die drei Schüler, die krank waren und die Prüfung nicht geschrieben haben.

</details>

**Aufgabe 2c:** Warum stehen bei diesen Schülern `NULL`-Werte?

*Deine Antwort:*

---

### Aufgabe 3: Online-Shop — JOIN + GROUP BY

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `products` | `product_id`, `product_name`, `category`, `price` |
| `orders` | `order_id`, `product_id`, `quantity`, `customer` |

Du kennst diese Tabellen bereits vom SQL-Aggregatfunktionen-Notebook.
Jetzt verknüpfen wir sie mit einem JOIN!

**3a)** Schreibe einen `INNER JOIN`, um jede Bestellung mit dem
zugehörigen Produktnamen und Preis anzuzeigen.
Zeige: `product_name`, `quantity`, `price`, `customer`.

**3b)** Berechne den **Gesamtumsatz pro Kategorie**.
Tipp: Umsatz = `price * quantity`. Nutze `SUM()` und `GROUP BY`.

In [ ]:
# AUFGABE 3a: INNER JOIN von orders und products.

result3a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 3a:\n', result3a)

<details>
<summary>Lösung anzeigen (3a)</summary>

```sql
SELECT products.product_name, orders.quantity, products.price, orders.customer
FROM orders
INNER JOIN products
  ON orders.product_id = products.product_id;
```

Ergebnis: 9 Zeilen — jede Bestellung jetzt mit Produktname und Preis.

</details>

In [ ]:
# AUFGABE 3b: Gesamtumsatz pro Kategorie (JOIN + GROUP BY).

result3b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 3b:\n', result3b)

<details>
<summary>Lösung anzeigen (3b)</summary>

```sql
SELECT products.category,
       SUM(orders.quantity * products.price) AS umsatz
FROM orders
INNER JOIN products
  ON orders.product_id = products.product_id
GROUP BY products.category;
```

Ergebnis: Electronics = 4165.00, Furniture = 2310.00.

</details>

---

### Aufgabe 4: Sportverein — Wer hat (nicht) am Turnier teilgenommen?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `mitglieder` | `mitglied_id`, `name`, `sportart` |
| `turnier_ergebnisse` | `mitglied_id`, `turnier`, `rang` |

Der Sportverein will eine Übersicht über alle Mitglieder und ihre
Turnierergebnisse. Nicht alle Mitglieder haben an einem Turnier
teilgenommen!

**4a)** Zeige alle Mitglieder mit ihren Turnierergebnissen.
Auch Mitglieder ohne Turnierergebnis sollen erscheinen.
Welchen JOIN-Typ brauchst du?

**4b)** Finde heraus: Welche Mitglieder haben noch **nie** an einem
Turnier teilgenommen?

In [ ]:
# AUFGABE 4a: Alle Mitglieder mit Turnierergebnissen (auch ohne!).

result4a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 4a:\n', result4a)

<details>
<summary>Lösung anzeigen (4a)</summary>

```sql
SELECT mitglieder.name, mitglieder.sportart,
       turnier_ergebnisse.turnier, turnier_ergebnisse.rang
FROM mitglieder
LEFT JOIN turnier_ergebnisse
  ON mitglieder.mitglied_id = turnier_ergebnisse.mitglied_id;
```

Ergebnis: 6 Zeilen. David und Finn haben `NULL` bei Turnier und Rang.

</details>

In [ ]:
# AUFGABE 4b: Mitglieder, die NIE an einem Turnier teilgenommen haben.

result4b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 4b:\n', result4b)

<details>
<summary>Lösung anzeigen (4b)</summary>

```sql
SELECT mitglieder.name, mitglieder.sportart
FROM mitglieder
LEFT JOIN turnier_ergebnisse
  ON mitglieder.mitglied_id = turnier_ergebnisse.mitglied_id
WHERE turnier_ergebnisse.turnier IS NULL;
```

Ergebnis: David (Schwimmen) und Finn (Fussball) — sie haben kein Turnierergebnis.

</details>

---

### Aufgabe 5: Bibliothek — Welche Bücher wurden nie ausgeliehen?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `buecher` | `buch_id`, `titel`, `autor`, `genre` |
| `ausleihen` | `ausleihe_id`, `buch_id`, `schueler`, `datum` |

Die Schulbibliothek will herausfinden, welche Bücher noch nie
ausgeliehen wurden, um sie besser zu bewerben.

**5a)** Zeige alle Bücher mit ihren Ausleihen an (auch Bücher
ohne Ausleihe).

**5b)** Finde die Bücher, die **noch nie** ausgeliehen wurden.

**5c)** Zähle, wie oft jedes Buch ausgeliehen wurde.
Sortiere absteigend nach der Ausleihzahl.
Tipp: Nutze `COUNT()`, `GROUP BY` und `ORDER BY`.

In [ ]:
# AUFGABE 5a: Alle Bücher mit ihren Ausleihen (auch ohne!).

result5a = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 5a:\n', result5a)

<details>
<summary>Lösung anzeigen (5a)</summary>

```sql
SELECT buecher.titel, buecher.autor,
       ausleihen.schueler, ausleihen.datum
FROM buecher
LEFT JOIN ausleihen
  ON buecher.buch_id = ausleihen.buch_id;
```

Ergebnis: 8 Zeilen. Eragon, Tintenherz und Rico haben `NULL` bei Schüler und Datum.

</details>

In [ ]:
# AUFGABE 5b: Bücher, die noch NIE ausgeliehen wurden.

result5b = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 5b:\n', result5b)

<details>
<summary>Lösung anzeigen (5b)</summary>

```sql
SELECT buecher.titel, buecher.autor
FROM buecher
LEFT JOIN ausleihen
  ON buecher.buch_id = ausleihen.buch_id
WHERE ausleihen.ausleihe_id IS NULL;
```

Ergebnis: Eragon, Tintenherz und Rico, Oskar und die Tieferschatten — diese drei wurden noch nie ausgeliehen.

</details>

In [ ]:
# AUFGABE 5c: Wie oft wurde jedes Buch ausgeliehen? (JOIN + GROUP BY + ORDER BY)

result5c = sql("""
-- Deine Abfrage hier
""")
print('Aufgabe 5c:\n', result5c)

<details>
<summary>Lösung anzeigen (5c)</summary>

```sql
SELECT buecher.titel,
       COUNT(ausleihen.ausleihe_id) AS anzahl_ausleihen
FROM buecher
LEFT JOIN ausleihen
  ON buecher.buch_id = ausleihen.buch_id
GROUP BY buecher.titel
ORDER BY anzahl_ausleihen DESC;
```

Ergebnis: Harry Potter und Gregs Tagebuch je 2×, Die drei ??? 1×, die restlichen drei 0×.

</details>

---

### Bonus: JOIN über drei Tabellen

Für diese Aufgabe erstellen wir eine zusätzliche Tabelle.
Führe zuerst die nächste Zelle aus.

In [ ]:
# Bonus-Setup (nicht verändern)
cursor.executescript("""
CREATE TABLE hausaufgaben (
    schueler_id INTEGER,
    punkte      INTEGER,
    FOREIGN KEY (schueler_id) REFERENCES klassenliste(schueler_id)
);
INSERT INTO hausaufgaben VALUES
    (1, 18),
    (2, 12),
    (3, 20),
    (4, 15),
    (5, 17),
    (6, 10);
""")
conn.commit()
print('Bonus-Tabelle erstellt!')

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `klassenliste` | `schueler_id`, `name`, `klasse` |
| `noten` | `schueler_id`, `note` |
| `hausaufgaben` | `schueler_id`, `punkte` |

Frau Müller will jetzt eine **Gesamtübersicht**: Name, Prüfungsnote
UND Hausaufgabenpunkte — für alle 6 Schüler.

Tipp: Du kannst zwei JOINs hintereinander schreiben:
```sql
SELECT ...
FROM tabelle_a
LEFT JOIN tabelle_b ON ...
LEFT JOIN tabelle_c ON ...;
```

In [ ]:
# BONUS: JOIN über drei Tabellen.

bonus = sql("""
-- Deine Abfrage hier
""")
print('Bonus:\n', bonus)

<details>
<summary>Lösung anzeigen</summary>

```sql
SELECT klassenliste.name, noten.note, hausaufgaben.punkte
FROM klassenliste
LEFT JOIN noten
  ON klassenliste.schueler_id = noten.schueler_id
LEFT JOIN hausaufgaben
  ON klassenliste.schueler_id = hausaufgaben.schueler_id;
```

Ergebnis: 6 Zeilen. Alle Schüler haben Hausaufgabenpunkte, aber nur 3 haben Noten. Bei den Kranken steht `NULL` in der Notenspalte.

</details>

---

## Reflexion

Beantworte die folgenden Fragen in dieser Zelle:

1. Erkläre in eigenen Worten: Was ist der Unterschied zwischen einem
   `INNER JOIN` und einem `LEFT JOIN`?

   *Deine Antwort:*

2. Nenne eine Situation aus deinem Alltag, in der du zwei Listen
   zusammenführen müsstest. Welchen JOIN-Typ würdest du verwenden?

   *Deine Antwort:*

3. In Aufgabe 3 hast du einen JOIN mit GROUP BY kombiniert.
   Warum brauchst du zuerst den JOIN, bevor du gruppieren kannst?

   *Deine Antwort:*

## 📊 Notebook-Übung: Tabellen verknüpfen mit pandas

:::{.callout-note}
## Wie man dieses Notebook lokal ausführt

1. Erstelle einen Ordner
2. Erstelle eine virtuelle Python-Umgebung  
   `python -m venv .venv`
3. Aktiviere die virtuelle Umgebung  
   `.venv\Scripts\activate`
4. Installiere Jupyter und pandas  
   `pip install jupyter pandas`
5. Lade diese Datei in deinen Ordner herunter
6. Starte den lokalen Jupyter-Server  
   `jupyter-lab`
:::

### Kontext

In der realen Welt sind Daten fast nie in einer einzigen Tabelle gespeichert.
Kundendaten liegen getrennt von Bestelldaten, Klassenlisten getrennt von Noten.
Um sinnvolle Analysen zu machen, müssen wir Tabellen **verknüpfen** — das
nennt man einen **JOIN** (oder in pandas: **merge**).

### Das Wichtigste auf einen Blick

| Was ich will | Code |
|---|---|
| Nur übereinstimmende Zeilen | `df1.merge(df2, on='schlüssel', how='inner')` |
| Alle Zeilen der linken Tabelle | `df1.merge(df2, on='schlüssel', how='left')` |
| Fehlende Werte finden | `ergebnis[ergebnis['spalte'].isna()]` |

**Dauer:** ca. 40–50 Minuten

**Voraussetzungen:** SQL-Grundlagen (`SELECT`, `FROM`, `WHERE`), grundlegende Python-Kenntnisse

---

### Setup

Führe die folgende Zelle aus, um die Daten vorzubereiten. **Nicht verändern.**

In [ ]:
import pandas as pd

# --- Daten für Aufgaben 1-2: Die Prüfungsliste ---

klassenliste = pd.DataFrame({
    'schueler_id': [1, 2, 3, 4, 5, 6],
    'name': ['Lea', 'Jan', 'Mira', 'Noah', 'Elena', 'Luca'],
    'klasse': ['2fP'] * 6
})

noten = pd.DataFrame({
    'schueler_id': [1, 2, 5],
    'note': [5.5, 4.0, 5.0]
})

# --- Daten für Aufgabe 3: Der Online-Shop (gleiche Daten wie im SQL-Notebook!) ---

products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5],
    'product_name': ['Laptop', 'Mouse', 'Desk', 'Chair', 'Headset'],
    'category': ['Electronics', 'Electronics', 'Furniture', 'Furniture', 'Electronics'],
    'price': [1200.00, 25.00, 450.00, 320.00, 80.00]
})

orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4, 5, 6, 7, 8, 9],
    'product_id': [1, 2, 3, 1, 4, 5, 2, 3, 5],
    'quantity': [2, 5, 1, 1, 3, 2, 8, 2, 1],
    'customer': ['Anna', 'Ben', 'Clara', 'Daniel', 'Anna', 'Ben', 'Clara', 'Evi', 'Anna']
})

# --- Daten für Aufgabe 4: Sportverein ---

mitglieder = pd.DataFrame({
    'mitglied_id': [1, 2, 3, 4, 5, 6],
    'name': ['Anna', 'Ben', 'Chiara', 'David', 'Eva', 'Finn'],
    'sportart': ['Fussball', 'Tennis', 'Fussball', 'Schwimmen', 'Tennis', 'Fussball']
})

turnier_ergebnisse = pd.DataFrame({
    'mitglied_id': [1, 2, 3, 5],
    'turnier': ['Kantonalmeisterschaft', 'Vereinsturnier', 'Kantonalmeisterschaft', 'Vereinsturnier'],
    'rang': [2, 5, 8, 3]
})

# --- Daten für Aufgabe 5: Nachhilfe-Matching ---

braucht_hilfe = pd.DataFrame({
    'name': ['Lea', 'Marco', 'Sara', 'Tim'],
    'fach': ['Mathe', 'Französisch', 'Mathe', 'Physik']
})

bietet_hilfe = pd.DataFrame({
    'name': ['Elena', 'Noah', 'Luca', 'Mira'],
    'fach': ['Mathe', 'Physik', 'Englisch', 'Mathe']
})

print('Alle Daten geladen. Viel Erfolg!')

---

### Aufgabe 1: INNER Merge — Wer hat die Prüfung geschrieben?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `klassenliste` | `schueler_id` (int), `name` (str), `klasse` (str) |
| `noten` | `schueler_id` (int), `note` (float) |

Frau Müller will nur die Schüler sehen, die die Prüfung **tatsächlich
geschrieben haben**, zusammen mit ihren Namen und Noten.

Benutze `klassenliste.merge(...)` mit `how='inner'` um die beiden
Tabellen über `schueler_id` zu verknüpfen.

**Erwartetes Ergebnis:** 3 Zeilen (Lea, Jan, Elena).

In [ ]:
# AUFGABE 1: Inner merge von klassenliste und noten auf 'schueler_id'.
# Dein Code hier:
result1 = ...
print(result1)

<details>
<summary>Lösung anzeigen</summary>

```python
result1 = klassenliste.merge(noten, on='schueler_id', how='inner')
print(result1)
```

</details>

### Aufgabe 2: LEFT Merge — Die vollständige Übersicht

Frau Müller braucht **alle 6 Schüler** in ihrer Übersicht — auch die,
die nicht an der Prüfung teilgenommen haben. Bei fehlenden Noten
soll `NaN` stehen.

**2a)** Führe einen LEFT merge durch.

**2b)** Finde heraus: Welche Schüler haben `NaN` in der Notenspalte?
Benutze dafür `ergebnis[ergebnis['note'].isna()]`.

**2c)** Beantworte: Warum stehen dort `NaN`-Werte?

In [ ]:
# AUFGABE 2a: Left merge von klassenliste und noten.
# Dein Code hier:
result2 = ...
print(result2)
print(f"\nAnzahl Zeilen: {len(result2)}")

<details>
<summary>Lösung anzeigen (2a)</summary>

```python
result2 = klassenliste.merge(noten, on='schueler_id', how='left')
print(result2)
print(f"\nAnzahl Zeilen: {len(result2)}")
```

Ergebnis: 6 Zeilen. Mira, Noah und Luca haben `NaN` in der Notenspalte, weil sie nicht in der `noten`-Tabelle vorkommen.

</details>

In [ ]:
# AUFGABE 2b: Schüler mit fehlenden Noten finden.
# Dein Code hier:
fehlend = ...
print(fehlend[['name']])

<details>
<summary>Lösung anzeigen (2b)</summary>

```python
result2 = klassenliste.merge(noten, on='schueler_id', how='left')
fehlend = result2[result2['note'].isna()]
print(fehlend[['name']])
```

Ausgabe: Mira, Noah, Luca.

</details>

**Aufgabe 2c:** Warum stehen bei diesen Schülern `NaN`-Werte?

*Deine Antwort:*

---

### Aufgabe 3: Der Online-Shop — Von SQL zu pandas

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `orders` | `order_id` (int), `product_id` (int), `quantity` (int), `customer` (str) |
| `products` | `product_id` (int), `product_name` (str), `category` (str), `price` (float) |

Du kennst diese Tabellen bereits aus dem SQL-Notebook!
Jetzt machst du dieselbe Analyse mit pandas statt SQL.

**3a)** Verknüpfe `orders` und `products` über `product_id` (Inner merge).
Zeige die ersten 5 Zeilen an.

**3b)** Berechne den **Gesamtumsatz pro Kategorie**.

Tipp: Erstelle nach dem Merge eine neue Spalte `umsatz = price * quantity`,
dann benutze `.groupby('category')['umsatz'].sum()`.

**Vergleich — so sähe es in SQL aus:**
```sql
SELECT category, SUM(price * quantity) AS umsatz
FROM orders
INNER JOIN products ON orders.product_id = products.product_id
GROUP BY category
```

In [ ]:
# AUFGABE 3a: Merge orders und products.
# Dein Code hier:
shop = ...
print(shop.head())

<details>
<summary>Lösung anzeigen (3a)</summary>

```python
shop = orders.merge(products, on='product_id', how='inner')
print(shop.head())
```

</details>

In [ ]:
# AUFGABE 3b: Gesamtumsatz pro Kategorie.
# Dein Code hier:

<details>
<summary>Lösung anzeigen (3b)</summary>

```python
shop = orders.merge(products, on='product_id', how='inner')
shop['umsatz'] = shop['price'] * shop['quantity']
umsatz_pro_kategorie = shop.groupby('category')['umsatz'].sum()
print(umsatz_pro_kategorie)
```

Erwartete Ausgabe:
```
category
Electronics    4025.0
Furniture      2310.0
```

</details>

---

### Aufgabe 4: Sportverein — Wer hat noch nie an einem Turnier teilgenommen?

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `mitglieder` | `mitglied_id` (int), `name` (str), `sportart` (str) |
| `turnier_ergebnisse` | `mitglied_id` (int), `turnier` (str), `rang` (int) |

Du hast zwei Tabellen:
- `mitglieder`: Alle Mitglieder des Vereins (6 Personen)
- `turnier_ergebnisse`: Ergebnisse von Turnieren (nur manche Mitglieder)

**4a)** Benutze einen LEFT merge auf `mitglied_id`, um alle Mitglieder
zu sehen — auch die ohne Turnierteilnahme.

**4b)** Welche Mitglieder haben **noch nie** an einem Turnier teilgenommen?
Filtere nach `NaN` in der Turnierspalte.

In [ ]:
# AUFGABE 4a: Left merge mitglieder und turnier_ergebnisse.
# Dein Code hier:
verein = ...
print(verein)

<details>
<summary>Lösung anzeigen (4a)</summary>

```python
verein = mitglieder.merge(turnier_ergebnisse, on='mitglied_id', how='left')
print(verein)
```

</details>

In [ ]:
# AUFGABE 4b: Mitglieder ohne Turnierteilnahme.
# Dein Code hier:

<details>
<summary>Lösung anzeigen (4b)</summary>

```python
verein = mitglieder.merge(turnier_ergebnisse, on='mitglied_id', how='left')
keine_teilnahme = verein[verein['turnier'].isna()]
print(keine_teilnahme[['name']])
```

Ausgabe: David, Finn.

</details>

---

### Aufgabe 5: Nachhilfe-Matching

**Tabellen:**

| Tabelle | Spalten |
|---|---|
| `braucht_hilfe` | `name` (str), `fach` (str) |
| `bietet_hilfe` | `name` (str), `fach` (str) |

An deiner Schule gibt es ein Nachhilfe-Programm:
- `braucht_hilfe`: Schüler, die Hilfe in einem Fach brauchen
- `bietet_hilfe`: Schüler, die Nachhilfe in einem Fach anbieten

**5a)** Benutze einen INNER merge auf `fach`, um passende Paare
zu finden (wer brauch Hilfe in einem Fach, das jemand anders anbietet?).

**Achtung:** Da beide Tabellen eine Spalte `name` haben, wird pandas
sie automatisch umbenennen in `name_x` und `name_y`. Das ist normal!

**5b)** Welches Fach hat **kein** Angebot (niemand bietet Nachhilfe an)?
Tipp: LEFT merge von `braucht_hilfe` auf `bietet_hilfe`, dann
nach `NaN` filtern.

In [ ]:
# AUFGABE 5a: Inner merge braucht_hilfe und bietet_hilfe auf 'fach'.
# Dein Code hier:
paare = ...
print(paare)

<details>
<summary>Lösung anzeigen (5a)</summary>

```python
paare = braucht_hilfe.merge(bietet_hilfe, on='fach', how='inner', suffixes=('_braucht', '_bietet'))
print(paare)
```

Ergebnis: Lea↔Elena (Mathe), Lea↔Mira (Mathe), Sara↔Elena (Mathe), Sara↔Mira (Mathe), Tim↔Noah (Physik).

</details>

In [ ]:
# AUFGABE 5b: Fächer ohne Nachhilfe-Angebot.
# Dein Code hier:

<details>
<summary>Lösung anzeigen (5b)</summary>

```python
alle_faecher = braucht_hilfe.merge(bietet_hilfe, on='fach', how='left', suffixes=('_braucht', '_bietet'))
ohne_angebot = alle_faecher[alle_faecher['name_bietet'].isna()]
print(ohne_angebot[['fach']].drop_duplicates())
```

Ausgabe: Französisch — niemand bietet dort Nachhilfe an.

</details>

---

### Bonusaufgabe: Drei Tabellen verknüpfen

**Tabellen:** `klassenliste` + `noten` (siehe Aufgabe 1) +

| Tabelle | Spalten |
|---|---|
| `hausaufgaben` | `schueler_id` (int), `ha_note` (float) |

Frau Müller hat jetzt auch eine Tabelle mit Hausaufgabennoten.
Verknüpfe alle drei Tabellen (`klassenliste`, `noten`, `hausaufgaben`)
und berechne für jeden Schüler eine Gesamtnote (Durchschnitt
von Prüfungsnote und Hausaufgabennote).

Tipp: Du kannst Merges verketten: `a.merge(b, ...).merge(c, ...)`

In [ ]:
# Bonus-Daten
hausaufgaben = pd.DataFrame({
    'schueler_id': [1, 2, 3, 4, 5, 6],
    'ha_note': [5.0, 4.5, 5.5, 4.0, 5.5, 3.5]
})

# Dein Code hier:

<details>
<summary>Lösung anzeigen (Bonus)</summary>

```python
gesamt = klassenliste.merge(noten, on='schueler_id', how='left') \
                     .merge(hausaufgaben, on='schueler_id', how='left')
gesamt['durchschnitt'] = gesamt[['note', 'ha_note']].mean(axis=1)
print(gesamt[['name', 'note', 'ha_note', 'durchschnitt']])
```

Schüler ohne Prüfungsnote bekommen nur die Hausaufgabennote als Durchschnitt (da `mean` `NaN` ignoriert).

</details>

---

### Reflexion

Beantworte die folgenden Fragen in dieser Zelle:

1. Erkläre in eigenen Worten: Was ist der Unterschied zwischen
   einem INNER merge und einem LEFT merge?

   *Deine Antwort:*

2. Nenne eine Situation aus deinem Alltag, in der du zwei Listen
   zusammenführen müsstest. Welchen Merge-Typ würdest du verwenden?

   *Deine Antwort:*

3. Was passiert, wenn die Schlüssel-Spalte in beiden Tabellen
   unterschiedliche Namen hat? (Tipp: Schau dir die Parameter
   `left_on` und `right_on` in der pandas-Dokumentation an.)

   *Deine Antwort:*